In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow
import mlflow.spark

catalog = "store_sales_catalog"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE {gold_schema}")


DataFrame[]

In [0]:
#check
df_fact = spark.table(f"{gold_schema}.fact_daily_sales")
display(df_fact.limit(10))


sales_fact_id,date_key,store_key,family_key,holiday_key,oil_key,date,sales,onpromotion,transactions,oil_price,is_holiday,created_at,updated_at
86897,20130218,1,8,null,null,2013-02-18,610.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86905,20130218,1,12,null,null,2013-02-18,92.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86913,20130218,1,16,null,null,2013-02-18,0.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86921,20130218,1,17,null,null,2013-02-18,0.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86929,20130218,1,18,null,null,2013-02-18,0.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86937,20130218,1,21,null,null,2013-02-18,1.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86945,20130218,1,26,null,null,2013-02-18,91.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86953,20130218,1,28,null,null,2013-02-18,0.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86961,20130218,1,30,null,null,2013-02-18,65.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z
86969,20130218,10,2,null,null,2013-02-18,0.0,0,1105,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z


In [0]:
# main fact table
fact = spark.table(f"{gold_schema}.fact_daily_sales").alias("f")

# dimension tables
dim_date    = spark.table(f"{gold_schema}.dim_date").alias("d")
dim_store   = spark.table(f"{gold_schema}.dim_store").alias("s")
dim_family  = spark.table(f"{gold_schema}.dim_family").alias("fa")
dim_holiday = spark.table(f"{gold_schema}.dim_holiday").alias("h")

print("fact rows:", fact.count())


fact rows: 3054348


In [0]:
#join fact and dims
joined = (
    fact
      .join(dim_date, "date_key",  "left")
      .join(dim_store, "store_key", "left")
      .join(dim_family, "family_key", "left")
      .join(dim_holiday, "holiday_key", "left")
)

joined.printSchema()
display(joined.limit(10))


root
 |-- holiday_key: long (nullable = true)
 |-- family_key: long (nullable = true)
 |-- store_key: long (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- sales_fact_id: long (nullable = false)
 |-- oil_key: long (nullable = true)
 |-- date: date (nullable = true)
 |-- sales: double (nullable = true)
 |-- onpromotion: integer (nullable = true)
 |-- transactions: integer (nullable = true)
 |-- oil_price: double (nullable = true)
 |-- is_holiday: boolean (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- store_nbr: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: s

holiday_key,family_key,store_key,date_key,sales_fact_id,oil_key,date,sales,onpromotion,transactions,oil_price,is_holiday,created_at,updated_at,date,day_of_week,week_of_year,month,month_name,year,is_weekend,created_at,store_nbr,city,state,type,cluster,created_at,family_name,created_at,date_key,type,locale,locale_name,description,transferred,is_holiday_effective,created_at
null,8,1,20130218,86897,null,2013-02-18,610.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z,2013-02-18,2,8,2,February,2013,false,2025-12-04T10:27:55.884Z,1,Quito,Pichincha,D,13,2025-12-04T10:27:06.216Z,CLEANING,2025-12-04T10:27:16.776Z,null,null,null,null,null,null,null,null
null,12,1,20130218,86905,null,2013-02-18,92.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z,2013-02-18,2,8,2,February,2013,false,2025-12-04T10:27:55.884Z,1,Quito,Pichincha,D,13,2025-12-04T10:27:06.216Z,FROZEN FOODS,2025-12-04T10:27:16.776Z,null,null,null,null,null,null,null,null
null,16,1,20130218,86913,null,2013-02-18,0.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z,2013-02-18,2,8,2,February,2013,false,2025-12-04T10:27:55.884Z,1,Quito,Pichincha,D,13,2025-12-04T10:27:06.216Z,HOME AND KITCHEN I,2025-12-04T10:27:16.776Z,null,null,null,null,null,null,null,null
null,17,1,20130218,86921,null,2013-02-18,0.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z,2013-02-18,2,8,2,February,2013,false,2025-12-04T10:27:55.884Z,1,Quito,Pichincha,D,13,2025-12-04T10:27:06.216Z,HOME AND KITCHEN II,2025-12-04T10:27:16.776Z,null,null,null,null,null,null,null,null
null,18,1,20130218,86929,null,2013-02-18,0.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z,2013-02-18,2,8,2,February,2013,false,2025-12-04T10:27:55.884Z,1,Quito,Pichincha,D,13,2025-12-04T10:27:06.216Z,HOME APPLIANCES,2025-12-04T10:27:16.776Z,null,null,null,null,null,null,null,null
null,21,1,20130218,86937,null,2013-02-18,1.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z,2013-02-18,2,8,2,February,2013,false,2025-12-04T10:27:55.884Z,1,Quito,Pichincha,D,13,2025-12-04T10:27:06.216Z,LAWN AND GARDEN,2025-12-04T10:27:16.776Z,null,null,null,null,null,null,null,null
null,26,1,20130218,86945,null,2013-02-18,91.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z,2013-02-18,2,8,2,February,2013,false,2025-12-04T10:27:55.884Z,1,Quito,Pichincha,D,13,2025-12-04T10:27:06.216Z,PERSONAL CARE,2025-12-04T10:27:16.776Z,null,null,null,null,null,null,null,null
null,28,1,20130218,86953,null,2013-02-18,0.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z,2013-02-18,2,8,2,February,2013,false,2025-12-04T10:27:55.884Z,1,Quito,Pichincha,D,13,2025-12-04T10:27:06.216Z,PLAYERS AND ELECTRONICS,2025-12-04T10:27:16.776Z,null,null,null,null,null,null,null,null
null,30,1,20130218,86961,null,2013-02-18,65.0,0,1742,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z,2013-02-18,2,8,2,February,2013,false,2025-12-04T10:27:55.884Z,1,Quito,Pichincha,D,13,2025-12-04T10:27:06.216Z,PREPARED FOODS,2025-12-04T10:27:16.776Z,null,null,null,null,null,null,null,null
null,2,10,20130218,86969,null,2013-02-18,0.0,0,1105,null,false,2025-12-04T10:42:57.210Z,2025-12-04T10:42:57.210Z,2013-02-18,2,8,2,February,2013,false,2025-12-04T10:27:55.884Z,10,Quito,Pichincha,C,15,2025-12-04T10:27:06.216Z,BABY CARE,2025-12-04T10:27:16.776Z,null,null,null,null,null,null,null,null


In [0]:
# pick out the columns we care about for modelling
df = joined.select(
    # from fact (use f.)
    F.col("f.date").alias("date"),          # for splitting / lag window
    F.col("f.date_key").alias("date_key"),
    F.col("f.store_key").alias("store_key"),
    F.col("f.family_key").alias("family_key"),
    F.col("f.sales").alias("sales"),        # target

    F.col("f.onpromotion").alias("onpromotion"),
    F.col("f.transactions").alias("transactions"),
    F.col("f.oil_price").alias("oil_price"),
    F.col("f.is_holiday").alias("is_holiday"),

    # from dim_date (d.)
    F.col("d.day_of_week").alias("day_of_week"),
    F.col("d.week_of_year").alias("week_of_year"),
    F.col("d.month").alias("month"),
    F.col("d.year").alias("year"),
    F.col("d.is_weekend").alias("is_weekend"),

    # from dim_store (s.)
    F.col("s.cluster").alias("cluster"),     # int, good simple proxy for store grouping i think

    # from dim_holiday (h.)
    F.col("h.is_holiday_effective").alias("is_holiday_effective")
)

print("rows before lag:", df.count())
display(df.limit(5))


rows before lag: 3054348


date,date_key,store_key,family_key,sales,onpromotion,transactions,oil_price,is_holiday,day_of_week,week_of_year,month,year,is_weekend,cluster,is_holiday_effective
2013-02-18,20130218,1,8,610.0,0,1742,null,false,2,8,2,2013,false,13,null
2013-02-18,20130218,1,12,92.0,0,1742,null,false,2,8,2,2013,false,13,null
2013-02-18,20130218,1,16,0.0,0,1742,null,false,2,8,2,2013,false,13,null
2013-02-18,20130218,1,17,0.0,0,1742,null,false,2,8,2,2013,false,13,null
2013-02-18,20130218,1,18,0.0,0,1742,null,false,2,8,2,2013,false,13,null


In [0]:
#Adding 7 day lag feature using date, remove the dates on the edges - i have to do this if we want this feature  
w = Window.partitionBy("store_key", "family_key").orderBy("date")

df = df.withColumn("lag_7", F.lag("sales", 7).over(w))

# toss rows that don't have a full 7-day history
df = df.dropna(subset=["lag_7"])

print("rows after lag_7:", df.count()) #we kinda do lose lose alot, should test results with and wihtout this feature 
display(df.limit(5))

rows after lag_7: 3041874


date,date_key,store_key,family_key,sales,onpromotion,transactions,oil_price,is_holiday,day_of_week,week_of_year,month,year,is_weekend,cluster,is_holiday_effective,lag_7
2013-01-08,20130108,1,9,384.0,0,1869,93.21,false,3,2,1,2013,false,13,null,0.0
2013-01-09,20130109,1,9,530.0,0,1910,93.08,false,4,2,1,2013,false,13,null,579.0
2013-01-10,20130110,1,9,345.0,0,1679,93.81,false,5,2,1,2013,false,13,null,453.0
2013-01-11,20130111,1,9,401.0,0,1813,93.6,false,6,2,1,2013,false,13,null,460.0
2013-01-12,20130112,1,9,404.0,0,1473,null,true,7,2,1,2013,true,13,null,464.0


In [0]:
# Train/Validation Split - Time Series Split
# Split chronologically to avoid data leakage (train dates < validation dates)

# Get date range
date_stats = df.agg(
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date")
).collect()[0]

min_date = date_stats["min_date"]
max_date = date_stats["max_date"]
total_days = (max_date - min_date).days

# Use ~80% for training, ~20% for validation
# Get unique dates and find 80% split point
date_list = sorted([row["date"] for row in df.select("date").distinct().collect()])
split_idx = int(len(date_list) * 0.8)
split_date_value = date_list[split_idx]

print(f"Date range: {min_date} to {max_date}")
print(f"Total days: {total_days}")
print(f"Split date (80%): {split_date_value}")
print(f"Training: {min_date} to {split_date_value}")
print(f"Validation: after {split_date_value} to {max_date}")

# Split the data
df_train = df.filter(F.col("date") <= split_date_value)
df_val = df.filter(F.col("date") > split_date_value)

train_count = df_train.count()
val_count = df_val.count()
total_count = train_count + val_count

print(f"\nTraining rows: {train_count}")
print(f"Validation rows: {val_count}")
print(f"Training %: {train_count / total_count * 100:.2f}%")


Date range: 2013-01-08 to 2017-08-15
Total days: 1680
Split date (80%): 2016-09-13
Training: 2013-01-08 to 2016-09-13
Validation: after 2016-09-13 to 2017-08-15

Training rows: 2434212
Validation rows: 607662
Training %: 80.02%


In [0]:
#Create model, can use VectorAssembler to combine features into a single vector column and then use mlflow to log the model - print rsms
# Fix feature columns (remove "dcoilwtico", use "oil_price" which is the correct column name)
feature_cols = [
    "day_of_week",
    "week_of_year",
    "month",
    "year",
    "is_weekend",
    "cluster",
    "onpromotion",
    "transactions",
    "oil_price",
    "is_holiday",
    "is_holiday_effective",
    "lag_7"
]

In [0]:
# Prepare training data: handle nulls and convert booleans to integers
print("Preparing training data...")

# Forward fill oil_price (time-series feature) - fill nulls with previous value per store+family
window_spec = Window.partitionBy("store_key", "family_key").orderBy("date").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_train_prep = df_train.withColumn(
    "oil_price", 
    F.last("oil_price", ignorenulls=True).over(window_spec)
)

# Fill remaining nulls with median (for oil_price if still null, and other numeric features)
# For transactions, fill with 0 if null
df_train_prep = df_train_prep.fillna({
    "oil_price": 0.0,  # Will be replaced with median if needed
    "transactions": 0,
    "onpromotion": 0,
    "cluster": 0
})

# Calculate median oil_price for final null fill
oil_median = df_train_prep.filter(F.col("oil_price").isNotNull()).select(F.percentile_approx("oil_price", 0.5).alias("median")).collect()[0]["median"]
if oil_median is None:
    oil_median = 0.0
df_train_prep = df_train_prep.fillna({"oil_price": float(oil_median)})

# Convert boolean columns to integers (0/1)
df_train_prep = df_train_prep.withColumn("is_weekend", F.col("is_weekend").cast("int"))
df_train_prep = df_train_prep.withColumn("is_holiday", F.col("is_holiday").cast("int"))
df_train_prep = df_train_prep.withColumn("is_holiday_effective", F.when(F.col("is_holiday_effective").isNull(), 0).otherwise(F.col("is_holiday_effective").cast("int")))

# Drop any remaining rows with nulls in critical features
df_train_prep = df_train_prep.dropna(subset=feature_cols + ["sales"])

print(f"Training rows after preprocessing: {df_train_prep.count()}")

Preparing training data...
Training rows after preprocessing: 2434212


In [0]:
# Prepare validation data with same transformations
df_val_prep = df_val.withColumn(
    "oil_price", 
    F.last("oil_price", ignorenulls=True).over(window_spec)
)
df_val_prep = df_val_prep.fillna({
    "oil_price": float(oil_median),
    "transactions": 0,
    "onpromotion": 0,
    "cluster": 0
})
df_val_prep = df_val_prep.withColumn("is_weekend", F.col("is_weekend").cast("int"))
df_val_prep = df_val_prep.withColumn("is_holiday", F.col("is_holiday").cast("int"))
df_val_prep = df_val_prep.withColumn("is_holiday_effective", F.when(F.col("is_holiday_effective").isNull(), 0).otherwise(F.col("is_holiday_effective").cast("int")))
df_val_prep = df_val_prep.dropna(subset=feature_cols + ["sales"])

print(f"Validation rows after preprocessing: {df_val_prep.count()}")

# Create VectorAssembler to combine features
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

# Transform data
df_train_vect = assembler.transform(df_train_prep)
df_val_vect = assembler.transform(df_val_prep)

Validation rows after preprocessing: 607662


In [0]:
# Train model - using GBTRegressor for baseline (can also try RandomForestRegressor)
print("\nTraining GBT Regressor model...")

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="sales",
    maxDepth=5,
    maxIter=20,
    seed=42
)

# Start MLflow experiment
# we use the default MLflow experiment
# (no path specified = default experiment, which always works and has full functionality)
# The default experiment works perfectly for all MLflow features: logging, tracking, model loading

print("Using default MLflow experiment (works perfectly for all requirements)")
# No need to set experiment - default will be used automatically
experiment_name = None

with mlflow.start_run(run_name="gbt_baseline_model") as run:
    # Train the model
    model = gbt.fit(df_train_vect)
    
    # Make predictions
    predictions_train = model.transform(df_train_vect)
    predictions_val = model.transform(df_val_vect)
    
    # Evaluate model
    evaluator_rmse = RegressionEvaluator(labelCol="sales", predictionCol="prediction", metricName="rmse")
    evaluator_mae = RegressionEvaluator(labelCol="sales", predictionCol="prediction", metricName="mae")
    evaluator_r2 = RegressionEvaluator(labelCol="sales", predictionCol="prediction", metricName="r2")
    
    rmse_train = evaluator_rmse.evaluate(predictions_train)
    mae_train = evaluator_mae.evaluate(predictions_train)
    r2_train = evaluator_r2.evaluate(predictions_train)
    
    rmse_val = evaluator_rmse.evaluate(predictions_val)
    mae_val = evaluator_mae.evaluate(predictions_val)
    r2_val = evaluator_r2.evaluate(predictions_val)
    
    # Log parameters
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("max_iter", 20)
    mlflow.log_param("features", ", ".join(feature_cols))
    mlflow.log_param("train_rows", df_train_prep.count())
    mlflow.log_param("val_rows", df_val_prep.count())
    
    # Log metrics
    mlflow.log_metric("train_rmse", rmse_train)
    mlflow.log_metric("train_mae", mae_train)
    mlflow.log_metric("train_r2", r2_train)
    mlflow.log_metric("val_rmse", rmse_val)
    mlflow.log_metric("val_mae", mae_val)
    mlflow.log_metric("val_r2", r2_val)
    
    # Log model with signature
    # For Databricks (serverless/shared clusters), need UC volume path for temp storage
    mlflow_tmp_path = f"/Volumes/{catalog}/raw/store_sales_vol/mlflow_tmp"
    
    from mlflow.models import infer_signature
    
    # Infer signature using Spark DataFrame (properly handles Vector types)
    # This is what matters for model registration and serving
    signature = infer_signature(
        df_train_vect.select("features"), 
        model.transform(df_train_vect.select("features"))
    )
    
    # Log model with signature only (signature is sufficient for Unity Catalog registration)
    mlflow.spark.log_model(
        model, 
        "model", 
        dfs_tmpdir=mlflow_tmp_path,
        signature=signature
    )
    
    # Store run_id for later use
    mlflow_run_id = run.info.run_id
    
    # Register model (optional - may not work due to Unity Catalog requirements)
    model_name = "store_sales_gbt_model"
    try:
        model_version = mlflow.register_model(f"runs:/{mlflow_run_id}/model", model_name)
        print(f"\n✓ Model registered as: {model_name}, version {model_version.version}")
    except Exception as e:
        # Model registration may fail - this is fine, model is still logged
        print(f"\nNote: Model registration not available: {str(e)[:200]}")
        print("Model is still fully logged and can be loaded using the run_id above")
    
    # Print metrics
    print("\n" + "="*50)
    print("MODEL EVALUATION METRICS")
    print("="*50)
    print(f"\nTRAINING SET:")
    print(f"  RMSE: {rmse_train:.4f}")
    print(f"  MAE:  {mae_train:.4f}")
    print(f"  R²:   {r2_train:.4f}")
    print(f"\nVALIDATION SET:")
    print(f"  RMSE: {rmse_val:.4f}")
    print(f"  MAE:  {mae_val:.4f}")
    print(f"  R²:   {r2_val:.4f}")
    print(f"\nMLflow Run ID: {mlflow_run_id}")
    print("="*50)

# Store predictions for later use
predictions_val_final = predictions_val.select(
    "date", "date_key", "store_key", "family_key", "sales", "prediction"
).withColumnRenamed("sales", "sales_actual").withColumnRenamed("prediction", "sales_predicted")

print("\nModel training and logging complete!")





Training GBT Regressor model...
Using default MLflow experiment (works perfectly for all requirements)


2025/12/05 20:38:37 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.2.2) contains a local version label (+databricks.connect.17.2.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2025/12/05 20:38:39 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-1e4d9e72-8c8b-40ab-a7a4-19/tmpnhsz557j/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
Registered model 'store_sales_gbt_model' already exists. Creating a new version of this model...



✓ Model registered as: store_sales_gbt_model, version 2

MODEL EVALUATION METRICS

TRAINING SET:
  RMSE: 415.7401
  MAE:  93.0529
  R²:   0.8376

VALIDATION SET:
  RMSE: 583.7710
  MAE:  140.0033
  R²:   0.8177

MLflow Run ID: 9beb4f7ce5e3477baa68270df5376a6b


Created version '2' of model 'store_sales_catalog.gold.store_sales_gbt_model'.



Model training and logging complete!


In [0]:
# Write Predictions Back to Gold Layer
# Create predictions table in Gold schema and write predictions with MERGE support

# Create predictions table if it doesn't exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_schema}.fact_daily_sales_predictions (
  date_key INT,
  store_key BIGINT,
  family_key BIGINT,
  date DATE,
  sales_actual DOUBLE,
  sales_predicted DOUBLE,
  prediction_error DOUBLE,
  prediction_date TIMESTAMP,
  model_version STRING,
  created_at TIMESTAMP,
  updated_at TIMESTAMP,
  PRIMARY KEY (date_key, store_key, family_key)
)
USING DELTA
PARTITIONED BY (date_key)
""")

print("Predictions table created/verified in Gold layer")

# Prepare predictions dataframe with all required columns
predictions_to_write = predictions_val_final.select(
    "date_key",
    "store_key", 
    "family_key",
    "date",
    "sales_actual",
    "sales_predicted"
).withColumn(
    "prediction_error", 
    F.col("sales_actual") - F.col("sales_predicted")
).withColumn(
    "prediction_date",
    F.current_timestamp()
).withColumn(
    "model_version",
    F.lit("gbt_baseline_v1")
).withColumn(
    "created_at",
    F.current_timestamp()
).withColumn(
    "updated_at",
    F.current_timestamp()
)

print(f"\nWriting {predictions_to_write.count()} predictions to Gold layer...")

# Write predictions using MERGE for incremental updates
predictions_to_write.createOrReplaceTempView("predictions_temp")

spark.sql(f"""
MERGE INTO {gold_schema}.fact_daily_sales_predictions AS tgt
USING predictions_temp AS src
ON tgt.date_key = src.date_key
  AND tgt.store_key = src.store_key
  AND tgt.family_key = src.family_key
WHEN MATCHED THEN
  UPDATE SET
    tgt.sales_actual = src.sales_actual,
    tgt.sales_predicted = src.sales_predicted,
    tgt.prediction_error = src.prediction_error,
    tgt.prediction_date = src.prediction_date,
    tgt.model_version = src.model_version,
    tgt.updated_at = src.updated_at
WHEN NOT MATCHED THEN
  INSERT (
    date_key,
    store_key,
    family_key,
    date,
    sales_actual,
    sales_predicted,
    prediction_error,
    prediction_date,
    model_version,
    created_at,
    updated_at
  )
  VALUES (
    src.date_key,
    src.store_key,
    src.family_key,
    src.date,
    src.sales_actual,
    src.sales_predicted,
    src.prediction_error,
    src.prediction_date,
    src.model_version,
    src.created_at,
    src.updated_at
  )
""")

# Verify the write
prediction_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {gold_schema}.fact_daily_sales_predictions").collect()[0]["cnt"]
print(f"\nPredictions successfully written to Gold layer!")
print(f"Total predictions in table: {prediction_count}")

# Display sample predictions
print("\nSample predictions:")
display(spark.sql(f"""
  SELECT 
    date,
    date_key,
    store_key,
    family_key,
    ROUND(sales_actual, 2) AS sales_actual,
    ROUND(sales_predicted, 2) AS sales_predicted,
    ROUND(prediction_error, 2) AS prediction_error,
    ROUND(ABS(prediction_error) / NULLIF(sales_actual, 0) * 100, 2) AS error_pct
  FROM {gold_schema}.fact_daily_sales_predictions
  ORDER BY date DESC, ABS(prediction_error) DESC
  LIMIT 20
"""))

Predictions table created/verified in Gold layer

Writing 607662 predictions to Gold layer...

Predictions successfully written to Gold layer!
Total predictions in table: 607662

Sample predictions:


date,date_key,store_key,family_key,sales_actual,sales_predicted,prediction_error,error_pct
2017-08-15,20170815,40,4,6845.0,3174.41,3670.59,53.62
2017-08-15,20170815,50,4,3661.0,6129.08,-2468.08,67.42
2017-08-15,20170815,1,4,1942.0,4149.31,-2207.31,113.66
2017-08-15,20170815,8,13,4035.0,6118.1,-2083.1,51.63
2017-08-15,20170815,45,31,4973.19,7029.81,-2056.62,41.35
2017-08-15,20170815,30,4,4052.0,1999.3,2052.7,50.66
2017-08-15,20170815,54,4,4332.0,2329.19,2002.81,46.23
2017-08-15,20170815,43,31,1743.11,3596.77,-1853.66,106.34
2017-08-15,20170815,11,4,4128.0,5955.07,-1827.07,44.26
2017-08-15,20170815,1,13,2508.0,4272.08,-1764.08,70.34


In [0]:
# Model Evaluation and Pipeline Integration
# Calculate detailed prediction vs actual differences and document pipeline integration

# Load predictions from Gold layer for evaluation
df_eval = spark.table(f"{gold_schema}.fact_daily_sales_predictions")

# Calculate aggregate evaluation metrics
eval_metrics = df_eval.agg(
    F.count("*").alias("total_predictions"),
    F.avg("prediction_error").alias("mean_error"),
    F.stddev("prediction_error").alias("std_error"),
    F.avg(F.abs("prediction_error")).alias("mean_absolute_error"),
    F.sqrt(F.avg(F.pow("prediction_error", 2))).alias("rmse"),
    F.percentile_approx(F.abs("prediction_error"), 0.5).alias("median_abs_error"),
    F.percentile_approx(F.abs("prediction_error"), 0.95).alias("p95_abs_error")
).collect()[0]

print("="*60)
print("PREDICTION EVALUATION METRICS (From Gold Layer)")
print("="*60)
print(f"Total Predictions: {eval_metrics['total_predictions']:,.0f}")
print(f"Mean Error: {eval_metrics['mean_error']:.4f}")
print(f"Std Dev Error: {eval_metrics['std_error']:.4f}")
print(f"Mean Absolute Error (MAE): {eval_metrics['mean_absolute_error']:.4f}")
print(f"Root Mean Squared Error (RMSE): {eval_metrics['rmse']:.4f}")
print(f"Median Absolute Error: {eval_metrics['median_abs_error']:.4f}")
print(f"95th Percentile Absolute Error: {eval_metrics['p95_abs_error']:.4f}")

# Calculate error by date (to see if errors are consistent over time)
print("\n" + "="*60)
print("ERROR ANALYSIS BY DATE")
print("="*60)
error_by_date = df_eval.groupBy("date").agg(
    F.count("*").alias("pred_count"),
    F.avg(F.abs("prediction_error")).alias("avg_abs_error"),
    F.sqrt(F.avg(F.pow("prediction_error", 2))).alias("rmse"),
    F.percentile_approx(F.abs("prediction_error"), 0.5).alias("median_error")
).orderBy("date")

display(error_by_date.limit(20))

# Calculate error distribution
print("\n" + "="*60)
print("ERROR DISTRIBUTION")
print("="*60)
error_dist = df_eval.select(
    F.when(F.abs("prediction_error") < 10, "0-10")
     .when(F.abs("prediction_error") < 50, "10-50")
     .when(F.abs("prediction_error") < 100, "50-100")
     .when(F.abs("prediction_error") < 500, "100-500")
     .otherwise("500+").alias("error_range"),
    "prediction_error"
).groupBy("error_range").agg(
    F.count("*").alias("count"),
    F.avg("prediction_error").alias("avg_error")
).orderBy("error_range")

display(error_dist)

# ============================================================================
# PIPELINE INTEGRATION DOCUMENTATION
# ============================================================================
print("\n" + "="*60)
print("PIPELINE INTEGRATION GUIDE")
print("="*60)
print("""
To incorporate this model into the daily pipeline:

1. LOAD MODEL FROM MLFLOW:
   ```python
   import mlflow.spark
   
   # Option A: If Model Registry is available
   # model_uri = "models:/store_sales_gbt_model/latest"  # or specific version
   # model = mlflow.spark.load_model(model_uri)
   
   # Option B: Load from run ID 
   # Get run_id from MLflow UI or from the printed run_id above
   run_id = "your_run_id_here"  # Replace with actual run_id
   model_uri = f"runs:/{run_id}/model"
   mlflow_tmp_path = "/Volumes/store_sales_catalog/raw/store_sales_vol/mlflow_tmp"
   model = mlflow.spark.load_model(model_uri, dfs_tmpdir=mlflow_tmp_path)
   
   # Option C: Load latest run from default experiment
   # Use default experiment (no path = default)
   experiment = mlflow.get_experiment_by_name(None)  # None = default experiment
   runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id], 
                            order_by=["start_time desc"], max_results=1)
   if not runs.empty:
       latest_run_id = runs.iloc[0]["run_id"]
       mlflow_tmp_path = "/Volumes/store_sales_catalog/raw/store_sales_vol/mlflow_tmp"
       model = mlflow.spark.load_model(f"runs:/{latest_run_id}/model", dfs_tmpdir=mlflow_tmp_path)
   ```

2. DAILY PREDICTION WORKFLOW:
   - Read new data from Gold layer (fact_daily_sales)
   - Apply same preprocessing as training:
     * Forward fill oil_price
     * Fill nulls with median/0
     * Convert booleans to integers
     * Create lag_7 feature (if needed)
   - Use VectorAssembler to create features
   - Generate predictions: predictions = model.transform(prepared_data)
   - Write predictions to fact_daily_sales_predictions using MERGE

3. BATCH INFERENCE EXAMPLE:
   ```python
   # Load latest model (using run ID)
   # Use default experiment (no path = default)
   experiment = mlflow.get_experiment_by_name(None)  # None = default experiment
   runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id], 
                            order_by=["start_time desc"], max_results=1)
   latest_run_id = runs.iloc[0]["run_id"]
   mlflow_tmp_path = "/Volumes/store_sales_catalog/raw/store_sales_vol/mlflow_tmp"
   model = mlflow.spark.load_model(f"runs:/{latest_run_id}/model", dfs_tmpdir=mlflow_tmp_path)
   
   # Alternative: If Model Registry is available
   # mlflow_tmp_path = "/Volumes/store_sales_catalog/raw/store_sales_vol/mlflow_tmp"
   # model = mlflow.spark.load_model("models:/store_sales_gbt_model/latest", dfs_tmpdir=mlflow_tmp_path)
   
   # Get new data (e.g., last 7 days)
   new_data = spark.table(f"{gold_schema}.fact_daily_sales")\
       .filter(F.col("date") >= F.current_date() - F.expr("INTERVAL 7 DAYS"))
   
   # Apply preprocessing (same as training)
   # ... preprocessing steps ...
   
   # Generate predictions
   predictions = model.transform(prepared_new_data)
   
   # Write to predictions table
   predictions.select(...).write.mode("append").saveAsTable(...)
   ```

4. STREAMING INFERENCE (if needed):
   - Use MLflow's spark_udf for real-time predictions
   - Integrate with Structured Streaming from Silver to Gold
   - Update predictions table incrementally

5. MODEL MONITORING:
   - Track prediction errors over time
   - Alert if RMSE/MAE exceeds thresholds
   - Retrain model periodically (e.g., monthly) with new data
   - Use MLflow to track model versions
   - Store run_id for each model version to track which model is in production

6. RETRAINING SCHEDULE:
   - Weekly: Retrain with latest data
   - Monthly: Full retrain with hyperparameter tuning
   - Quarterly: Feature engineering review and model architecture updates
""")

# Example: Load model and make predictions on a sample
print("\n" + "="*60)
print("EXAMPLE: Loading Model and Making Predictions")
print("="*60)

try:
    # In Free Edition, search all runs (works without specifying experiment_id)
    # Filter by run name to find our specific model run
    print("Searching for latest model run...")
    
    # Search all runs, ordered by start time (most recent first)
    # Filter by run name "gbt_baseline_model" to find our specific run
    runs = mlflow.search_runs(
        filter_string="tags.mlflow.runName = 'gbt_baseline_model'",
        order_by=["start_time desc"],
        max_results=1
    )
    
    # If no run found with that name, try getting the latest run without filter
    if runs.empty:
        print("Run with name 'gbt_baseline_model' not found, searching for latest run...")
        runs = mlflow.search_runs(
            order_by=["start_time desc"],
            max_results=1
        )
    
    if not runs.empty:
        latest_run_id = runs.iloc[0]["run_id"]
        run_name = runs.iloc[0].get("tags.mlflow.runName", "unnamed")
        print(f"Found run: {run_name} (ID: {latest_run_id})")
        
        # Try to load the model (need UC volume path for Free Edition)
        mlflow_tmp_path = f"/Volumes/{catalog}/raw/store_sales_vol/mlflow_tmp"
        loaded_model = mlflow.spark.load_model(f"runs:/{latest_run_id}/model", dfs_tmpdir=mlflow_tmp_path)
        print(f"✓ Model loaded successfully from run: {latest_run_id}")
        
        # Get a small sample for demonstration
        sample_data = df_val_vect.limit(10)
        sample_predictions = loaded_model.transform(sample_data)
        
        print("\nSample predictions on validation data:")
        display(sample_predictions.select(
            "date", "store_key", "family_key", "sales", "prediction"
        ).withColumn("error", F.col("sales") - F.col("prediction")))
    else:
        print("Note: No runs found. Make sure Cell 7 has been executed to train and log the model.")
        print("You can also load the model manually using the run_id from Cell 7 output.")
except Exception as e:
    print(f"Note: Could not load model for demo - {e}")
    print("Model is still logged in MLflow and can be accessed via MLflow UI")
    print("You can load it manually using the run_id printed in Cell 7")

print("\n" + "="*60)
print("EVALUATION COMPLETE")
print("="*60)
print("\nNext Steps:")
print("1. Review prediction errors and identify patterns")
print("2. Consider feature engineering improvements (e.g., more lag features, rolling averages)")
print("3. Experiment with different models (RandomForest, XGBoost, etc.)")
print("4. Set up automated daily prediction pipeline")
print("5. Monitor model performance and retrain as needed")


INFO:py4j.clientserver:Received command c on object id p0


PREDICTION EVALUATION METRICS (From Gold Layer)
Total Predictions: 607,662
Mean Error: -19.5208
Std Dev Error: 583.4450
Mean Absolute Error (MAE): 140.0033
Root Mean Squared Error (RMSE): 583.7710
Median Absolute Error: 11.1865
95th Percentile Absolute Error: 618.5141

ERROR ANALYSIS BY DATE


date,pred_count,avg_abs_error,rmse,median_error
2016-09-14,1782,94.13141666752641,329.58329982417814,6.986547783573734
2016-09-15,1782,64.44346826605964,218.23169429993624,7.063501471361761
2016-09-16,1782,77.73753541053284,265.38949598846415,8.231926446416463
2016-09-17,1782,119.75088453540297,539.4855455516307,9.572473705286676
2016-09-18,1782,155.96759058258567,747.002906738894,8.091357235548585
2016-09-19,1782,80.80042977232208,288.97619916072017,6.091357235548585
2016-09-20,1782,70.78628765420468,239.58604625699397,7.015750085328026
2016-09-21,1782,76.51830946528369,297.8429121508324,6.93328753684773
2016-09-22,1782,67.53208500590686,242.9440394441512,6.793071408022786
2016-09-23,1782,86.28046326374464,366.8379646215137,7.063501471361761



ERROR DISTRIBUTION


error_range,count,avg_error
0-10,292625,-1.8849671279274158
10-50,146504,-3.070752608036142
100-500,76049,-44.19239611835305
50-100,55657,-11.797435560418633
500+,36827,-185.82018987012626



PIPELINE INTEGRATION GUIDE

To incorporate this model into the daily pipeline:

1. LOAD MODEL FROM MLFLOW:
   ```python
   import mlflow.spark
   
   # Option A: If Model Registry is available (may not be in Free Edition)
   # model_uri = "models:/store_sales_gbt_model/latest"  # or specific version
   # model = mlflow.spark.load_model(model_uri)
   
   # Option B: Load from run ID (works in Free Edition)
   # Get run_id from MLflow UI or from the printed run_id above
   run_id = "your_run_id_here"  # Replace with actual run_id
   model_uri = f"runs:/{run_id}/model"
   mlflow_tmp_path = "/Volumes/store_sales_catalog/raw/store_sales_vol/mlflow_tmp"
   model = mlflow.spark.load_model(model_uri, dfs_tmpdir=mlflow_tmp_path)
   
   # Option C: Load latest run from default experiment (Free Edition compatible)
   # Use default experiment (no path = default)
   experiment = mlflow.get_experiment_by_name(None)  # None = default experiment
   runs = mlflow.search_runs(experiment_ids=[experiment

date,store_key,family_key,sales,prediction,error
2016-09-14,1,9,773.0,1015.266026719494,-242.266026719494
2016-09-15,1,9,674.0,671.5385937190952,2.461406280904839
2016-09-16,1,9,834.0,666.7338314831367,167.26616851686333
2016-09-17,1,9,789.0,1004.189963219841,-215.18996321984105
2016-09-18,1,9,419.0,325.7990839172996,93.20091608270042
2016-09-19,1,9,751.0,1021.4902054643262,-270.4902054643262
2016-09-20,1,9,758.0,675.648561096905,82.35143890309496
2016-09-21,1,9,895.0,672.1750034400526,222.8249965599474
2016-09-22,1,9,623.0,674.9064081953123,-51.90640819531234
2016-09-23,1,9,737.0,1012.0847441959879,-275.0847441959879



EVALUATION COMPLETE

Next Steps:
1. Review prediction errors and identify patterns
2. Consider feature engineering improvements (e.g., more lag features, rolling averages)
3. Experiment with different models (RandomForest, XGBoost, etc.)
4. Set up automated daily prediction pipeline
5. Monitor model performance and retrain as needed
